# 04 — Weibull AFT (improved): years until a bridge leaves Good

**Primary objective:** estimate the median **remaining years** until a bridge first drops from Good to Fair/Poor, given how long it has already stayed Good (`entry_duration`).

**What changed from the earlier Weibull notebook** (chosen in Notebook 13 on the 2016 historical validation, not on the 2021 test set):
1. Added the deck, superstructure, substructure, culvert and overall condition ratings from each bridge's first Good year.
2. Log-transformed the skewed traffic and length inputs and added a curved (squared) age term.

Notebook 13 validation C-index: 0.6156 (earlier setup) to 0.6427 (this setup). State was tested and dropped because it added nothing once ratings were included.

**Evaluation:** ranking C-index on the **entire** aligned test set. It measures ranking, **not** accuracy in years, and does not adjust for different entry durations. Bridges first Good in 1992 have an unknown true start.

In [1]:
from pathlib import Path
import time, warnings
import numpy as np
import pandas as pd
from lifelines.utils import concordance_index

warnings.filterwarnings("ignore", category=FutureWarning)

# Project folders (same logic as the other notebooks).
PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
if PROJECT.name == "bridge_time_primary_notebooks":
    PROJECT = PROJECT.parent
FINAL = PROJECT / "data" / "final"
PROCESSED = PROJECT / "data" / "processed"
RESULTS = PROJECT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

RARE_MIN = 200    # categories rarer than this are grouped as "other"

from lifelines import WeibullAFTFitter

PENALIZER = 0.01
LOGS = ["ADT_029", "STRUCTURE_LEN_MT_049"]    # skewed inputs that are log-transformed
print("Project directory:", PROJECT)

Project directory: /Users/annie/Downloads/bridge-deterioration/bridge_time_primary_notebooks/notebooks/final notebooks


## 1. Load data and add condition ratings

In [2]:
EXTRA_SRC = ["bridge_id", "STATE_CODE_001", "DECK_COND_058", "SUPERSTRUCTURE_COND_059",
             "SUBSTRUCTURE_COND_060", "CULVERT_COND_062", "condition_rating"]
RATING_COLS = ["deck_rating", "super_rating", "sub_rating", "culvert_rating", "min_rating"]

def add_baseline_extras(df):
    # Attach state and ratings recorded in each bridge's first Good year (its start_year).
    parts = []
    for yr, grp in df.groupby("start_year"):
        snap = pd.read_parquet(PROCESSED / f"nbi_{int(yr)}_clean.parquet", columns=EXTRA_SRC)
        snap = snap.drop_duplicates("bridge_id")
        parts.append(snap[snap["bridge_id"].isin(grp["bridge_id"])])
    extras = pd.concat(parts, ignore_index=True).rename(columns={
        "STATE_CODE_001": "state", "DECK_COND_058": "deck_rating",
        "SUPERSTRUCTURE_COND_059": "super_rating", "SUBSTRUCTURE_COND_060": "sub_rating",
        "CULVERT_COND_062": "culvert_rating", "condition_rating": "min_rating"})
    out = df.merge(extras, on="bridge_id", how="left", validate="one_to_one")
    out["state"] = out["state"].astype("string")
    for c in RATING_COLS:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out


train_data = pd.read_parquet(FINAL / "bridge_survival_train.parquet")
test_data = pd.read_parquet(FINAL / "bridge_survival_test_aligned.parquet")

# Add the condition ratings recorded in each bridge's first Good year.
train_data = add_baseline_extras(train_data)
test_data = add_baseline_extras(test_data)

BASE_NUM = ["bridge_age", "ADT_029", "STRUCTURE_LEN_MT_049"]
BASE_CAT = ["STRUCTURE_KIND_043A", "STRUCTURE_TYPE_043B"]

print("Training records:", f"{len(train_data):,}", "| events:", f"{int(train_data['event'].sum()):,}")
print("Testing records:", f"{len(test_data):,}", "| events:", f"{int(test_data['event'].sum()):,}")
print("\nOverall rating in first Good year (train):")
print(train_data["min_rating"].value_counts(dropna=False).sort_index().to_string())
print("Missing ratings (train / test):", int(train_data["min_rating"].isna().sum()), "/", int(test_data["min_rating"].isna().sum()))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/annie/Downloads/bridge-deterioration/bridge_time_primary_notebooks/notebooks/final notebooks/data/final/bridge_survival_train.parquet'

## 2. Prepare model inputs

In [ ]:
def make_design(tr, other, num, cat=(), rating_cols=(), log_cols=(), age_sq=False, scale=False):
    # Build model inputs from `tr`, applying exactly the same fitted choices to `other`.
    X_tr = pd.DataFrame(index=tr.index)
    X_ot = pd.DataFrame(index=other.index)
    cont = []
    for c in list(num) + list(rating_cols):
        t = pd.to_numeric(tr[c], errors="coerce")
        o = pd.to_numeric(other[c], errors="coerce")
        if c in rating_cols:                         # remember where a rating was not recorded
            X_tr[c + "_na"] = t.isna().astype(int)
            X_ot[c + "_na"] = o.isna().astype(int)
        med = t.median()
        med = 0.0 if not np.isfinite(med) else med
        t, o = t.fillna(med), o.fillna(med)
        if age_sq and c == "bridge_age":
            X_tr["bridge_age_sq"], X_ot["bridge_age_sq"] = t ** 2, o ** 2
            cont.append("bridge_age_sq")
        if c in log_cols:
            t, o = np.log1p(t.clip(lower=0)), np.log1p(o.clip(lower=0))
        X_tr[c], X_ot[c] = t, o
        cont.append(c)
    if scale:                                        # standardise using training values only
        for c in cont:
            mu, sd = X_tr[c].mean(), X_tr[c].std()
            if sd and np.isfinite(sd):
                X_tr[c] = (X_tr[c] - mu) / sd
                X_ot[c] = (X_ot[c] - mu) / sd
    for c in cat:
        t = tr[c].astype("string").fillna("__missing__")
        o = other[c].astype("string").fillna("__missing__")
        counts = t.value_counts()
        keep = set(counts[counts >= RARE_MIN].index)
        t = t.where(t.isin(keep), "__other__")
        o = o.where(o.isin(keep), "__other__")
        cats = sorted(set(t.astype(str)) | {"__other__", "__missing__"})
        for series, X in ((t, X_tr), (o, X_ot)):
            d = pd.get_dummies(
                pd.Series(pd.Categorical(series.astype(str), categories=cats), index=series.index),
                prefix=c, drop_first=True, dtype=int)
            for col in d.columns:
                X[col] = d[col]
    X_ot = X_ot[X_tr.columns]
    keep_cols = [c for c in X_tr.columns if X_tr[c].nunique() > 1]   # drop constant columns
    return X_tr[keep_cols].astype(float), X_ot[keep_cols].astype(float)

RATING_COLS_USED = RATING_COLS

X_train, X_test = make_design(
    train_data, test_data, num=BASE_NUM, cat=BASE_CAT, rating_cols=RATING_COLS_USED,
    log_cols=LOGS, age_sq=True, scale=True)

print("Model inputs:", X_train.shape[1])
print(list(X_train.columns))
assert np.isfinite(X_train.to_numpy()).all() and np.isfinite(X_test.to_numpy()).all()
assert len(X_test) == len(test_data)

Model inputs: 41
['bridge_age_sq', 'bridge_age', 'ADT_029', 'STRUCTURE_LEN_MT_049', 'deck_rating_na', 'deck_rating', 'super_rating_na', 'super_rating', 'sub_rating_na', 'sub_rating', 'culvert_rating_na', 'culvert_rating', 'min_rating', 'STRUCTURE_KIND_043A_1', 'STRUCTURE_KIND_043A_2', 'STRUCTURE_KIND_043A_3', 'STRUCTURE_KIND_043A_4', 'STRUCTURE_KIND_043A_5', 'STRUCTURE_KIND_043A_6', 'STRUCTURE_KIND_043A_7', 'STRUCTURE_KIND_043A_8', 'STRUCTURE_KIND_043A_9', 'STRUCTURE_KIND_043A___missing__', 'STRUCTURE_TYPE_043B_01', 'STRUCTURE_TYPE_043B_02', 'STRUCTURE_TYPE_043B_03', 'STRUCTURE_TYPE_043B_04', 'STRUCTURE_TYPE_043B_05', 'STRUCTURE_TYPE_043B_06', 'STRUCTURE_TYPE_043B_07', 'STRUCTURE_TYPE_043B_08', 'STRUCTURE_TYPE_043B_09', 'STRUCTURE_TYPE_043B_10', 'STRUCTURE_TYPE_043B_11', 'STRUCTURE_TYPE_043B_12', 'STRUCTURE_TYPE_043B_19', 'STRUCTURE_TYPE_043B_20', 'STRUCTURE_TYPE_043B_21', 'STRUCTURE_TYPE_043B_22', 'STRUCTURE_TYPE_043B___missing__', 'STRUCTURE_TYPE_043B___other__']


## 3. Train the Weibull AFT model

In [ ]:
fit_df = X_train.assign(duration=train_data["duration"].to_numpy(float),
                        event=train_data["event"].to_numpy(int))
weibull_model = WeibullAFTFitter(penalizer=PENALIZER)
weibull_model.fit(fit_df, duration_col="duration", event_col="event")

print("Weibull AFT fitted.")
print("Log-likelihood:", round(weibull_model.log_likelihood_, 2))
print("Training C-index:", round(weibull_model.concordance_index_, 4))

Weibull AFT fitted.
Log-likelihood: -939745.6
Training C-index: 0.7105


## 4. Median remaining years for **all** test bridges

For Weibull, `S(t) = exp(-(t/lambda)^rho)`. The median remaining time `m` solves `S(entry + m) / S(entry) = 0.5`. Predictions past the last observed training event time are extrapolated and flagged.

In [ ]:
entry = test_data["entry_duration"].to_numpy(dtype=float)
test_duration = test_data["test_duration"].to_numpy(dtype=float)
test_event = test_data["event"].to_numpy(dtype=int)
assert np.isfinite(entry).all() and (entry >= 0).all()
assert np.isfinite(test_duration).all() and (test_duration > 0).all()
assert np.isin(test_event, [0, 1]).all()

# lambda = time at S = exp(-1); t50 = unconditional median. Together they give rho per bridge.
lam = weibull_model.predict_percentile(X_test, p=np.exp(-1)).to_numpy(float)
t50 = weibull_model.predict_median(X_test).to_numpy(float)
rho = np.log(np.log(2.0)) / np.log(t50 / lam)
assert np.isfinite(lam).all() and (lam > 0).all() and np.isfinite(rho).all() and (rho > 0).all()

with np.errstate(over="ignore", invalid="ignore"):
    total_median = lam * np.exp(np.logaddexp(rho * np.log(np.maximum(entry, 1e-300) / lam), np.log(np.log(2.0))) / rho)
median_remaining_years = np.maximum(total_median - entry, 0.0)
assert np.isfinite(median_remaining_years).all()

last_training_event = float(train_data.loc[train_data["event"].astype(bool), "duration"].max())
beyond_support = total_median > last_training_event

predictions = test_data[["bridge_id", "entry_duration", "test_duration", "event"]].copy().reset_index(drop=True)
predictions["median_remaining_years"] = median_remaining_years
predictions["median_total_years_since_first_good"] = total_median
predictions["beyond_last_training_event"] = beyond_support
predictions["weibull_scale"] = lam
predictions["risk_score"] = -lam            # higher = earlier predicted deterioration (same convention in all models)
predictions.to_csv(RESULTS / "time_weibull_aft_improved.csv", index=False)

print("Test bridges:", f"{len(predictions):,}", "| medians produced:", f"{np.isfinite(median_remaining_years).sum():,}")
print("Predicted remaining years (descriptive, not error):")
print(predictions["median_remaining_years"].describe(percentiles=[.1, .25, .5, .75, .9]).round(2).to_string())
print("Beyond last observed training event (", last_training_event, "years):", f"{int(beyond_support.sum()):,}", "/", f"{len(predictions):,}")
print("Saved:", RESULTS / "time_weibull_aft_improved.csv")

Test bridges: 171,766 | medians produced: 171,766
Predicted remaining years (descriptive, not error):
count    171766.00
mean         23.86
std          12.68
min           2.13
10%           9.46
25%          13.72
50%          21.30
75%          32.65
90%          43.76
max          79.38
Beyond last observed training event ( 28.0 years): 132,178 / 171,766
Saved: /Users/annie/Downloads/bridge-deterioration/results/time_weibull_aft_improved.csv


## 5. Full-test ranking evaluation

Two rankings, both on **all** test bridges: the conditional median remaining years (uses entry time) and the Weibull scale (does not). Neither is a measure of error in years, and neither adjusts for delayed entry.

In [ ]:
c_remaining = float(concordance_index(test_duration, median_remaining_years, test_event))
c_scale = float(concordance_index(test_duration, lam, test_event))

print("Evaluated test bridges:", f"{len(test_data):,}", "| events:", f"{int(test_event.sum()):,}")
print("FULL-test C-index, conditional median remaining years:", round(c_remaining, 4))
print("FULL-test C-index, Weibull scale (risk_score):         ", round(c_scale, 4))
print("Training C-index:", round(float(weibull_model.concordance_index_), 4))

metrics = pd.DataFrame([{
    "model": "Weibull AFT (improved)",
    "inputs": int(X_train.shape[1]),
    "train_records": len(train_data), "test_records": len(test_data), "test_events": int(test_event.sum()),
    "training_c_index": float(weibull_model.concordance_index_),
    "full_test_c_index_remaining_years": c_remaining,
    "full_test_c_index_risk_score": c_scale,
    "predictions_beyond_training_support": int(beyond_support.sum()),
}])
metrics.to_csv(RESULTS / "weibull_aft_improved_metrics.csv", index=False)
print("Saved:", RESULTS / "weibull_aft_improved_metrics.csv")

Evaluated test bridges: 171,766 | events: 22,396
FULL-test C-index, conditional median remaining years: 0.6276
FULL-test C-index, Weibull scale (risk_score):          0.6168
Training C-index: 0.7105
Saved: /Users/annie/Downloads/bridge-deterioration/results/weibull_aft_improved_metrics.csv


## 6. What the model learned

In [ ]:
lam_summary = weibull_model.summary.loc["lambda_"]
print("Largest effects on the time scale (exp(coef) < 1 = deteriorates sooner; inputs are standardised):")
display(lam_summary[["coef", "exp(coef)", "p"]].assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False).drop(columns="abs_coef").head(15).round(4))
print("Weibull shape rho:", round(float(np.exp(weibull_model.params_.loc[("rho_", "Intercept")])), 4),
      "(above 1 = the chance of leaving Good rises the longer a bridge stays Good)")

Largest effects on the time scale (exp(coef) < 1 = deteriorates sooner; inputs are standardised):


,coef,exp(coef),p
covariate,,,
Intercept,3.1761,23.9530,0.0000
STRUCTURE_KIND_043A_7,-0.4504,0.6374,0.0000
STRUCTURE_KIND_043A_9,-0.4366,0.6462,0.0000
bridge_age,-0.3813,0.6830,0.0000
STRUCTURE_KIND_043A_3,-0.3313,0.7180,0.0000
STRUCTURE_TYPE_043B_19,0.2969,1.3457,0.0000
STRUCTURE_KIND_043A_5,0.2450,1.2776,0.0000
STRUCTURE_TYPE_043B_05,-0.2239,0.7994,0.0000
STRUCTURE_TYPE_043B_08,-0.2098,0.8108,0.0001


Weibull shape rho: 1.2158 (above 1 = the chance of leaving Good rises the longer a bridge stays Good)
